# **LIBRARIES**

In [ ]:
import requests
import json

from functools import reduce

from libpysal.weights import Rook, Queen, KNN
from spopt.region import RegionKMeansHeuristic, AZP, MaxPHeuristic, WardSpatial, Skater

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import pairwise as skm
from sklearn import metrics

from sklearn.decomposition import PCA, KernelPCA
from sklearn.cluster import KMeans, AgglomerativeClustering
import scipy.cluster.hierarchy as shc

import numpy as np
import pandas as pd
import geopandas as gpd

import mapclassify

import seaborn
import matplotlib.pyplot as plt

import warnings
warnings.filterwarnings("ignore")

# **SOCIOECONOMIC/DEMOGRAPHIC DATA**

## **Functions**

In [ ]:
def get_tables(table_file):
        
    tables = pd.read_csv('data/tables/tables_' + table_file + '.csv')
    table_codes = tables.TABLES.values
    table_codes = ','.join(table_codes)
    table_names = list(tables.NAMES)
        
    return table_codes, table_names
    
    
def get_api_key():
    
    with open('census_api_key.txt') as key:
        api_key = key.read().strip()    
        
    return api_key
    
    
def get_data(table_file, year, dsource='acs/acs5'):
        
    table_codes, table_names = get_tables(table_file)
    api_key = get_api_key()   

    base_url = f'https://api.census.gov/data/{year}/{dsource}'
    data_url = f'?get={table_codes}'
    key_url = f'&key={api_key}'    
    #geography_url = f'&for=tract:*&in=county:005,047,061,081,085&in=state:36'
    geography_url = f'&for=tract:*&in=county:047&in=state:36'
    
    url = base_url + data_url + geography_url + key_url

    json = requests.get(url).json()
    df = pd.DataFrame(json[1:], columns=json[0])
    
    table_names.append('State')
    table_names.append('County')
    table_names.append('Tract')
    df.columns = table_names
      
    #df['year'] = year
    #year = df.pop('year')
    #df.insert(1, 'year', year)
    
    df['GEOID'] = df['State'] + df['County'] + df['Tract']
    df = df.drop(columns=['State', 'County', 'Tract'])
    
    df = df.apply(pd.to_numeric, errors='ignore')
    df['GEOID'] = df['GEOID'].astype(str)
    
    return df

## **Tract Geometries**

In [ ]:
tracts = gpd.read_file('https://www2.census.gov/geo/tiger/TIGER2019/TRACT/tl_2019_36_tract.zip')
tracts = tracts[tracts.COUNTYFP == '047']
tracts = gpd.GeoDataFrame(tracts[['GEOID', 'geometry']])
tracts = tracts.to_crs('EPSG:2263')

boroughs = gpd.read_file('data/boroughs/boroughs.shp')
boroughs = boroughs.to_crs('EPSG:2263')
brooklyn = boroughs[boroughs.BoroName == 'Brooklyn']

tracts = tracts.clip(brooklyn)

## **Variables**

In [ ]:
population = get_data('acs_population', 2019)

In [ ]:
race = get_data('acs_race', 2019)
race = race[['Total Population', 'Not Hispanic or Latino: White Alone', 'Not Hispanic or Latino: Black Alone',
             'Not Hispanic or Latino: Asian Alone', 'Hispanic or Latino', 'GEOID']]
race.columns = ['Total', 'White', 'Black', 'Asian', 'Latino', 'GEOID']
race.loc[:, 'White':'Latino'] = race.loc[:, 'White':'Latino'].div(race['Total'], axis=0) * 100
race = race.drop(columns=['Total'])
race.replace([np.inf, -np.inf, np.nan], 0, inplace=True)

In [ ]:
age = get_data('acs_age', 2019)
age['Children'] = (age.loc[:, 'Male: Under 5 Years':'Male: 15 to 17 Years'].sum(axis=1) + age.loc[:, 'Female: Under 5 Years':'Female: 15 to 17 Years'].sum(axis=1)).div(age['Total Population']) * 100
age['Seniors'] = (age.loc[:, 'Male: 65 and 66 Years':'Male: 85 Years and Over'].sum(axis=1) + age.loc[:, 'Female: 65 and 66 Years':'Female: 85 Years and Over'].sum(axis=1)).div(age['Total Population']) * 100
age.replace([np.inf, -np.inf, np.nan], 0, inplace=True)
age = age[['Children', 'Seniors', 'GEOID']]

In [ ]:
citizenship = get_data('acs_citizenship', 2019)
citizenship.iloc[:, :-1] = citizenship.iloc[:, :-1].div(citizenship['Total Population'], axis=0) * 100
citizenship = citizenship.drop(columns=['Total Population'])

In [ ]:
nativity = get_data('acs_nativity', 2019)
nativity.iloc[:, :-1] = nativity.iloc[:, :-1].div(nativity['Total Population'], axis=0) * 100
nativity['Born in Europe'] = nativity['Foreign born and naturalized from Europe'] + nativity['Foreign born and not a citizen from Europe']
nativity['Born in Asia'] = nativity['Foreign born and naturalized from Asia'] + nativity['Foreign born and not a citizen from Asia']
nativity['Born in Africa'] = nativity['Foreign born and naturalized from Africa'] + nativity['Foreign born and not a citizen from Africa']
nativity['Born in Oceania'] = nativity['Foreign born and naturalized from Oceania'] + nativity['Foreign born and not a citizen from Oceania']
nativity['Born in Latin America'] = nativity['Foreign born and naturalized from Latin America'] + nativity['Foreign born and not a citizen from Latin America']
nativity['Born in Northern America'] = nativity['Foreign born and naturalized from Northern America'] + nativity['Foreign born and not a citizen from Northern America']
nativity = nativity[['Foreign born', 'Born in Europe', 'Born in Asia', 'Born in Africa', 'Born in Oceania', 'Born in Latin America', 'Born in Northern America', 'GEOID']]

In [ ]:
educational_attainment = get_data('acs_educational_attainment', 2019)
educational_attainment.iloc[:, :-1] = educational_attainment.iloc[:, :-1].div(educational_attainment['Total Population'], axis=0) * 100
educational_attainment['Less than HS Diploma'] = educational_attainment.loc[:, 'No School Completed':'12th Grade - No Diploma'].sum(axis=1)
educational_attainment['HS Diploma'] = educational_attainment.loc[:, 'Regular High School Diploma':'GED or Alternative Credential'].sum(axis=1)
educational_attainment['Some College'] = educational_attainment.loc[:, 'Some College - Less Than 1 Year':'Associates Degree'].sum(axis=1)
educational_attainment['College Degree'] = educational_attainment['Bachelors Degree']
educational_attainment['Graduate Degree'] = educational_attainment.loc[:, 'Masters Degree':'Doctorate Degree'].sum(axis=1)
educational_attainment = educational_attainment.loc[:, 'GEOID':'Graduate Degree']

In [ ]:
median_income = get_data('acs_median_income', 2019)

In [ ]:
income = get_data('acs_income_bands', 2019)
income.iloc[:, :-1] = income.iloc[:, :-1].div(income['Total Households'], axis=0) * 100
income = income.drop(columns=['Total Households'])
income.columns = income.columns.str.replace('$', '')

In [ ]:
median_rent = get_data('acs_median_rents', 2019)

In [ ]:
rent = get_data('acs_gross_rent', 2019)
rent.iloc[:, :-1] = rent.iloc[:, :-1].div(rent['Total Population'], axis=0) * 100
rent = rent.drop(columns=['Total Population', 'Total with Cash Rent', 'No cash rent'])
rent.columns = 'Rent ' + rent.columns
rent.columns = rent.columns.str.replace('$','')
rent = rent.rename(columns={'Rent GEOID':'GEOID'})

In [ ]:
poverty_rate = get_data('acs_family_poverty_rate', 2019)
poverty_rate['Family Poverty Rate'] = poverty_rate['# of Families Below Poverty Line'] / (poverty_rate['# of Families Below Poverty Line'] + poverty_rate['# of Families Above Poverty Line']) * 100
poverty_rate = poverty_rate[['Family Poverty Rate', 'GEOID']]

In [ ]:
tenure = get_data('acs_tenure', 2019)
tenure.loc[:, 'Total Housing Units':'Renter Occupied'] = tenure.loc[:, 'Total Housing Units':'Renter Occupied'].div(tenure['Total Housing Units'], axis=0) * 100
tenure = tenure[['Owner Occupied', 'Renter Occupied', 'GEOID']]
tenure.replace([np.inf, -np.inf, np.nan], 0, inplace=True)

In [ ]:
housing_typology = get_data('acs_housing_typology', 2019)
housing_typology.iloc[:, :-1] = housing_typology.iloc[:, :-1].div(housing_typology['Total Housing Units'], axis=0) * 100
housing_typology = housing_typology.drop(columns=['Total Housing Units', 'Mobile Home', 'Boat, RV, Van, etc.'])

In [ ]:
housing_age = get_data('acs_housing_age', 2019)
housing_age.iloc[:, :-1] = housing_age.iloc[:, :-1].div(housing_age['Total Housing Units'], axis=0) * 100
housing_age = housing_age.drop(columns=['Total Housing Units'])
housing_age.columns = 'Housing Built ' + housing_age.columns
housing_age = housing_age.rename(columns={'Housing Built GEOID':'GEOID'})

## **Combine Variables & Tracts**

In [ ]:
dfs = [population, race, age, citizenship, nativity, educational_attainment, median_income, income, median_rent, rent, poverty_rate, tenure, housing_typology, housing_age, tracts]
df = reduce(lambda  left,right: pd.merge(left,right,on=['GEOID'], how='outer'), dfs)

geoid = df.pop('GEOID')
df.insert(0, 'GEOID', geoid)

df = gpd.GeoDataFrame(df)

df.dropna(subset=['geometry'], how='all', inplace=True)
#df[df.isnull().any(axis=1)]
df = df[df['Total Population'] != 0]
df = df.drop(columns=['Total Population'])
df = df.replace(-666666666, 0)
df = df.fillna(0)

df.to_file('data/brooklyn_data.geojson', driver='GeoJSON')

In [ ]:
variables = df.columns.tolist()[1:-1]

## **Choropleths**

In [ ]:
def plot_variable_choropleths(nrows, ncols, variables, savepath):
    
    fig, ax = plt.subplots(nrows, ncols, figsize=(20,40))
    
    pooled = mapclassify.Pooled(df[variables], classifier="Quantiles", k=10)
    
    if nrows != 1 & ncols != 1:
        ax = ax.flatten()

        for i, variable in enumerate(variables):
            df.plot(column=variable, scheme="UserDefined", classification_kwds={"bins": pooled.global_classifier.bins}, legend=True, ax=ax[i])
            ax[i].set_title(variables[i], fontsize=20, fontweight='bold')
            ax[i].set_axis_off()

    elif nrows == 1 & ncols == 1:
        
        df.plot(column=variables[0], scheme="UserDefined", classification_kwds={"bins": pooled.global_classifier.bins}, legend=True, ax=ax)
        ax.set_title(variables[0], fontsize=20, fontweight='bold')
        ax.set_axis_off()
    
    plt.tight_layout()
    plt.savefig(savepath, dpi=200)

In [ ]:
plot_variable_choropleths(4, 2, ['Housing Built Before 1940', 'Housing Built 1950',
                                 'Housing Built 1960', 'Housing Built 1970', 'Housing Built 1980',
                                 'Housing Built 1990', 'Housing Built 2000', 'Housing Built 2014'], 'images/choropleths_variables_building_age.jpg')

In [ ]:
plot_variable_choropleths(4, 2, ['Detached Single Family Home', '2 Attached Homes', '2 Apartments',
                                 '3 to 4 Apartments', '5 to 9 Apartments', '10 to 19 Apartments',
                                 '20 to 49 Apartments', '50 or More Apartments'], 'images/choropleths_variables_building_typology.jpg')

In [ ]:
plot_variable_choropleths(1, 2, ['Renter Occupied', 'Owner Occupied'], 'images/choropleths_variables_tenure.jpg')

In [ ]:
plot_variable_choropleths(1, 1, ['Family Poverty Rate'], 'images/choropleths_variables_poverty_rate.jpg')

In [ ]:
plot_variable_choropleths(1, 1, ['Median Gross Rent'], 'images/choropleths_variables_median_rent.jpg')

In [ ]:
plot_variable_choropleths(1, 1, ['Median Income'], 'images/choropleths_variables_median_income.jpg')

In [ ]:
plot_variable_choropleths(2, 2, ['Less than HS Diploma', 'HS Diploma', 'College Degree', 'Graduate Degree'], 'images/choropleths_variables_education.jpg')

In [ ]:
plot_variable_choropleths(2, 2, ['Born in Europe', 'Born in Asia', 'Born in Africa', 'Born in Latin America'], 'images/choropleths_variables_nativity.jpg')

In [ ]:
plot_variable_choropleths(1, 2, ['Foreign born', 'Not a US citizen'], 'images/choropleths_variables_citizenship.jpg')

In [ ]:
plot_variable_choropleths(1, 2, ['Children', 'Seniors'], 'images/choropleths_variables_age.jpg')

In [ ]:
plot_variable_choropleths(2, 2, ['White', 'Black', 'Asian', 'Latino'], 'images/choropleths_variables_race.jpg')

# **PCA**

## **Data Standardization**

In [ ]:
scaled = StandardScaler().fit_transform(df[variables])
df_scaled = pd.DataFrame(scaled, columns=variables)

## **PCA Fit**

In [ ]:
pca = PCA()
principal_components = pca.fit_transform(scaled)

principal_df = pd.DataFrame(data=principal_components)
principal_df.columns = ['PC' + str(i) for i in range(1, principal_df.shape[1]+1)]

## **Scree Plot**

In [ ]:
per_var = np.round(pca.explained_variance_ratio_ * 100, decimals = 1)
labels = ['PC' + str(x) for x in range(1, len(per_var)+1)]

plt.figure(figsize=(40,20))

plt.bar(x=range(1,len(per_var)+1), height=per_var, tick_label=labels)
plt.xticks(rotation = 45)
plt.ylabel('Percentage of Explained Variance', fontsize=20)
plt.xlabel('Principal Component', fontsize=20)
plt.title('Scree Plot', fontsize=30)

plt.savefig('images/pca_screeplot.png', dpi=150)

In [ ]:
principal_df = principal_df.iloc[:, :np.where(pca.explained_variance_ >= 1)[0][-1]]

## **Loadings**

In [ ]:
loadings = pd.DataFrame(pca.components_.T, columns=['PC' + str(i) for i in range(1, principal_components.shape[1]+1)], index=variables)

In [ ]:
def print_top10_loadings(principal_component_number):

    loading_scores = pd.Series(pca.components_[principal_component_number], index=variables)
    sorted_loading_scores = loading_scores.abs().sort_values(ascending=False)

    top_10_variables= sorted_loading_scores[0:10].index.values
    print(loading_scores[top_10_variables].to_latex())

In [ ]:
print_top10_loadings(0)
print_top10_loadings(1)
print_top10_loadings(2)
print_top10_loadings(3)
print_top10_loadings(4)

## **Scatterplot**

In [ ]:
plt.figure(figsize=(15,15))

plt.scatter(principal_df['PC1'], principal_df['PC2']);
plt.xlabel('Principal Component 1', fontsize=20)
plt.ylabel('Principal Component 2', fontsize=20)

plt.savefig('images/pca_scatterplot.png', dpi=150)

## **Choropleths**

In [ ]:
principal_df['GEOID'] = df['GEOID']
principal_df['geometry'] = df['geometry']
principal_df = gpd.GeoDataFrame(principal_df)

pca_variables = principal_df.columns.tolist()[:-2]

In [ ]:
# fig, ax = plt.subplots(5, 4, figsize=(20,20))
# ax = ax.flatten()

# for i in range(1,21):
#     column = 'PC' + str(i)
#     principal_df.plot(column=column, ax=ax[i-1])
#     ax[i-1].set_title(column)
#     ax[i-1].set_axis_off()
    
# plt.savefig('images/pca_choropleths.jpg', dpi=300)

In [ ]:
fig, ax = plt.subplots(1, 4, figsize=(40,40))
ax = ax.flatten()

for i in range(1,5):
    column = 'PC' + str(i)
    principal_df.plot(column=column, ax=ax[i-1])
    ax[i-1].set_title(column)
    ax[i-1].set_axis_off()
    
plt.savefig('images/pca_choropleths.jpg', dpi=300)

# **KERNEL PCA**

## **Kernel PCA Fit**

In [ ]:
kernel_pca = KernelPCA(n_components=40, kernel="rbf")
kernel_principal_components = kernel_pca.fit_transform(scaled)

kernel_principal_df = pd.DataFrame(data=kernel_principal_components)
kernel_principal_df.columns = ['PC' + str(i) for i in range(1, kernel_principal_components.shape[1]+1)]

## **Scatterplot**

In [ ]:
plt.figure(figsize=(15,15))

plt.scatter(kernel_principal_df['PC1'], kernel_principal_df['PC2']);
plt.xlabel('Principal Component 1', fontsize=20)
plt.ylabel('Principal Component 2', fontsize=20)

plt.savefig('images/pca_scatterplot_kernel.png', dpi=150)

## **Choropleths**

In [ ]:
kernel_principal_df['GEOID'] = df['GEOID']
kernel_principal_df['geometry'] = df['geometry']
kernel_principal_df = gpd.GeoDataFrame(kernel_principal_df)

kernel_pca_variables = kernel_principal_df.columns.tolist()[:-2]

In [ ]:
# fig, ax = plt.subplots(5, 4, figsize=(20,20))
# ax = ax.flatten()

# for i in range(1,21):
#     column = 'PC' + str(i)
#     kernel_principal_df.plot(column=column, ax=ax[i-1])
#     ax[i-1].set_title(column)
#     ax[i-1].set_axis_off()
    
# plt.savefig('images/pca_choropleths_kernel.jpg', dpi=200)

In [ ]:
fig, ax = plt.subplots(1, 4, figsize=(40,40))
ax = ax.flatten()

for i in range(1,5):
    column = 'PC' + str(i)
    kernel_principal_df.plot(column=column, ax=ax[i-1])
    ax[i-1].set_title(column)
    ax[i-1].set_axis_off()
    
plt.savefig('images/pca_choropleths_kernel.jpg', dpi=300)

# **SPATIAL WEIGHTS**

In [ ]:
df = df.reset_index(drop=True)

w = Rook.from_dataframe(df)
#w = Queen.from_dataframe(df)
#w = KNN.from_dataframe(df, k=5)

In [ ]:
fig, ax = plt.subplots(1,1, figsize=(20,20))

w.plot(df, edge_kws=dict(linewidth=.5),
              node_kws=dict(s=0), ax=ax)
df.plot(ax=ax, alpha=.6)
ax.set_axis_off()

plt.savefig('images/spatial_weights_rook.jpg', dpi=300)

# **CLUSTERING**

## **K-Means**

### **Elbow Graphs**

In [ ]:
def make_elbow_graph(dataframe, dataframe_variables, savepath):
    
    K = range(1, 50)
    sse = []
    
    for k in K:
        kmeans = KMeans(n_clusters=k)
        kmeans = kmeans.fit(dataframe[dataframe_variables])
        sse.append(kmeans.inertia_)
        
    plt.figure(figsize=(20,20))
    plt.plot(range(1,50), sse, '-o', c='maroon')
    plt.xlabel('Number of Clusters')
    plt.ylabel('SSE')
    plt.savefig(savepath, dpi=150)

In [ ]:
make_elbow_graph(df_scaled, variables, 'images/kmeans_elbow_graph_standardized.jpg')

In [ ]:
make_elbow_graph(principal_df, pca_variables, 'images/kmeans_elbow_graph_pca.jpg')

In [ ]:
make_elbow_graph(kernel_principal_df, kernel_pca_variables, 'images/kmeans_elbow_graph_kernel.jpg')

### **Scores & Optimal Number of Clusters**

In [ ]:
def get_kmeans_scores(dataframe, variables_list):
    
    kmeans_clusters = pd.DataFrame()

    for i in range(2, 31):
        kmeans = KMeans(n_clusters=i, random_state=42)
        kmeans = kmeans.fit(dataframe[variables_list])

        kmeans_clusters['Clusters ' + str(i)] = kmeans.labels_
        
    
    silhouette_scores = []
    ch_scores = []
    db_scores = []

    for number_clusters in ["Clusters " + str(i) for i in range(2,31)]:
        silhouette_score = metrics.silhouette_score(dataframe[variables_list], kmeans_clusters[number_clusters])
        silhouette_scores.append((number_clusters, silhouette_score))

        ch_score = metrics.calinski_harabasz_score(dataframe[variables_list], kmeans_clusters[number_clusters])
        ch_scores.append((number_clusters, ch_score))
        
        db_score = metrics.davies_bouldin_score(dataframe[variables_list], kmeans_clusters[number_clusters])
        db_scores.append((number_clusters, db_score))


    silhouette_scores = pd.DataFrame(silhouette_scores, columns=['Number of Clusters', 'Silhouette Score']).set_index('Number of Clusters')
    ch_scores = pd.DataFrame(ch_scores, columns=['Number of Clusters', 'CH Score']).set_index('Number of Clusters')
    db_scores = pd.DataFrame(db_scores, columns=['Number of Clusters', 'DB Score']).set_index('Number of Clusters')
    
    scores = silhouette_scores.merge(ch_scores, on=['Number of Clusters']).merge(db_scores, on=['Number of Clusters'])

    return scores

In [ ]:
kmeans_scores_scaled = get_kmeans_scores(df_scaled, variables)
kmeans_scores_pca = get_kmeans_scores(principal_df, pca_variables)
kmeans_scores_kernel = get_kmeans_scores(kernel_principal_df, kernel_pca_variables)

In [ ]:
def top_scores(df, dataset_name, cluster_technique_name):

    top_db_score = pd.DataFrame(df.sort_values('DB Score', ascending=True).reset_index().iloc[0,:][['Number of Clusters', 'DB Score']]).T
    top_db_score['Score Type'] = 'Davies-Bouldin'
    top_db_score['Number of Clusters'] = top_db_score['Number of Clusters'].str.replace('Clusters ', '')
    top_db_score = top_db_score.rename(columns={'DB Score':'Best Score'})

    top_si_score = pd.DataFrame(df.sort_values('Silhouette Score', ascending=False).reset_index().iloc[0,:][['Number of Clusters', 'Silhouette Score']]).T
    top_si_score['Score Type'] = 'Silhouette'
    top_si_score['Number of Clusters'] = top_si_score['Number of Clusters'].str.replace('Clusters ', '')
    top_si_score = top_si_score.rename(columns={'Silhouette Score':'Best Score'})

    top_ch_score = pd.DataFrame(df.sort_values('CH Score', ascending=False).reset_index().iloc[0,:][['Number of Clusters', 'CH Score']]).T
    top_ch_score['Score Type'] = 'Calinski-Harabasz'
    top_ch_score['Number of Clusters'] = top_ch_score['Number of Clusters'].str.replace('Clusters ', '')
    top_ch_score = top_ch_score.rename(columns={'CH Score':'Best Score'})

    top_score = pd.concat([top_db_score, top_si_score, top_ch_score])
    top_score = top_score.rename(columns={'Number of Clusters':'Optimal Number of Clusters'})
    top_score['Clustering Algorithm'] = cluster_technique_name
    top_score['Dataset'] = dataset_name

    return top_score

kmeans_scaled_top_scores = top_scores(kmeans_scores_scaled, 'Standardized', 'K-Means')
kmeans_pca_top_scores = top_scores(kmeans_scores_pca, 'PCA', 'K-Means')
kmeans_kernel_top_scores = top_scores(kmeans_scores_kernel, 'Kernel PCA', 'K-Means')
kmeans_top_scores = pd.concat([kmeans_scaled_top_scores, kmeans_pca_top_scores, kmeans_kernel_top_scores])
print(kmeans_top_scores.to_latex(index=False))

In [ ]:
def plot_scores(dfs, savepath):
    
    fig, axes = plt.subplots(1, 3, figsize=(30,10))
    axes.flatten()
    
    ax = axes[0]
    ax.plot(range(2,31), dfs[0]['Silhouette Score'], label='Standardized');
    ax.plot(range(2,31), dfs[1]['Silhouette Score'], label='PCA');
    ax.plot(range(2,31), dfs[2]['Silhouette Score'], label='Kernel PCA');
    ax.legend();
    ax.set_ylabel('Score')
    ax.set_xlabel('Number of Clusters')
    ax.set_title('Silhouette Score');

    ax = axes[1]
    ax.plot(range(2,31), dfs[0]['CH Score'], label='Standardized');
    ax.plot(range(2,31), dfs[1]['CH Score'], label='PCA');
    ax.plot(range(2,31), dfs[2]['CH Score'], label='Kernel PCA');
    ax.legend();
    ax.set_ylabel('Score')
    ax.set_xlabel('Number of Clusters')
    ax.set_title('Calinski Harabasz Score');
    
    ax = axes[2]
    ax.plot(range(2,31), dfs[0]['DB Score'], label='Standardized');
    ax.plot(range(2,31), dfs[1]['DB Score'], label='PCA');
    ax.plot(range(2,31), dfs[2]['DB Score'], label='Kernel PCA');
    ax.legend();
    ax.set_ylabel('Score')
    ax.set_xlabel('Number of Clusters')
    ax.set_title('Davies Bouldin Score');
    
    plt.savefig(savepath, dpi=200)
    
plot_scores([kmeans_scores_scaled, kmeans_scores_pca, kmeans_scores_kernel], 'images/scores_kmeans.jpg')

### **Fit KMeans**

In [ ]:
def fit_kmeans(n_clusters, dataframe, dataframe_variables, labels_name):
    
    kmeans = KMeans(n_clusters=n_clusters, random_state=42)
    kmeans = kmeans.fit(dataframe[dataframe_variables])
    
    df[labels_name] = kmeans.labels_
    kernel_principal_df[labels_name] = kmeans.labels_
    principal_df[labels_name] = kmeans.labels_

In [ ]:
fit_kmeans(7, kernel_principal_df, kernel_pca_variables, 'kernel_pca_kmeans_7')
fit_kmeans(3, principal_df, pca_variables, 'pca_kmeans_3')
fit_kmeans(3, df_scaled, variables, 'scaled_kmeans_3')

### **Maps**

In [ ]:
f, axs = plt.subplots(3, 1, figsize=(40, 40))
axs.flatten()

ax = axs[0]
df.plot(
    column="kernel_pca_kmeans_7",
    categorical=True,
    cmap="Set3",
    legend=False,
    linewidth=0,
    ax=ax,
)
ax.set_axis_off()
ax.set_title("K-Means for Kernel PCA ($k=7$)", fontsize=30)

ax = axs[1]
df.plot(
    column="pca_kmeans_3",
    categorical=True,
    cmap="Set3",
    legend=False,
    linewidth=0,
    ax=ax,
)
ax.set_axis_off()
ax.set_title("K-Means for PCA ($k=3$)", fontsize=30)

ax = axs[2]
df.plot(
    column="scaled_kmeans_3",
    categorical=True,
    cmap="Set3",
    legend=False,
    linewidth=0,
    ax=ax,
)
ax.set_axis_off()
ax.set_title("K-Means for Standardized ($k=3$)", fontsize=30)

plt.tight_layout()

plt.savefig('images/maps_kmeans.jpg', dpi=200)

## **Agglomerative Hierarchical (AHC)**

### **Heatmap & Dendrogram**

In [ ]:
plt.figure(figsize=(30,30))
seaborn.clustermap(principal_df[pca_variables], cbar_pos=None, yticklabels=False);
plt.savefig('images/ahc_heatmap_pca.jpg', dpi=200)

In [ ]:
plt.figure(figsize=(30,30));
seaborn.clustermap(kernel_principal_df[kernel_pca_variables[:20]], cbar_pos=None, yticklabels=False);
plt.savefig('images/ahc_heatmap_kernel.jpg', dpi=200)

In [ ]:
plt.figure(figsize=(50,20))
dend = shc.dendrogram(shc.linkage(df_scaled[variables], method='ward'), no_labels=True);
plt.savefig('images/ahc_dendrogram_standardized.jpg', dpi=200)

In [ ]:
plt.figure(figsize=(50,20))
dend = shc.dendrogram(shc.linkage(principal_df[pca_variables], method='ward'), no_labels=True);
plt.savefig('images/ahc_dendrogram_pca.jpg', dpi=200)

In [ ]:
plt.figure(figsize=(50,20))
dend = shc.dendrogram(shc.linkage(kernel_principal_df[kernel_pca_variables], method='ward'), no_labels=True);
plt.savefig('images/ahc_dendrogram_kernel.jpg', dpi=200)

### **Scores & Optimal Number of Clusters**

In [ ]:
def get_ahc_scores(dataframe, variables_list):
    
    ahc_clusters = pd.DataFrame()

    for i in range(2, 31):
        ahc = AgglomerativeClustering(linkage="ward", n_clusters=i)
        ahc = ahc.fit(dataframe[variables_list])

        ahc_clusters['Clusters ' + str(i)] = ahc.labels_
        
    
    silhouette_scores = []
    ch_scores = []
    db_scores = []

    for number_clusters in ["Clusters " + str(i) for i in range(2,31)]:
        silhouette_score = metrics.silhouette_score(dataframe[variables_list], ahc_clusters[number_clusters])
        silhouette_scores.append((number_clusters, silhouette_score))

        ch_score = metrics.calinski_harabasz_score(dataframe[variables_list], ahc_clusters[number_clusters])
        ch_scores.append((number_clusters, ch_score))
        
        db_score = metrics.davies_bouldin_score(dataframe[variables_list], ahc_clusters[number_clusters])
        db_scores.append((number_clusters, db_score))


    silhouette_scores = pd.DataFrame(silhouette_scores, columns=['Number of Clusters', 'Silhouette Score']).set_index('Number of Clusters')
    ch_scores = pd.DataFrame(ch_scores, columns=['Number of Clusters', 'CH Score']).set_index('Number of Clusters')
    db_scores = pd.DataFrame(db_scores, columns=['Number of Clusters', 'DB Score']).set_index('Number of Clusters')
    
    scores = silhouette_scores.merge(ch_scores, on=['Number of Clusters']).merge(db_scores, on=['Number of Clusters'])

    return scores

In [ ]:
ahc_scores_scaled = get_ahc_scores(df_scaled, variables)
ahc_scores_pca = get_ahc_scores(principal_df, pca_variables)
ahc_scores_kernel = get_ahc_scores(kernel_principal_df, kernel_pca_variables)

In [ ]:
def top_scores(df, dataset_name, cluster_technique_name):

    top_db_score = pd.DataFrame(df.sort_values('DB Score', ascending=True).reset_index().iloc[0,:][['Number of Clusters', 'DB Score']]).T
    top_db_score['Score Type'] = 'Davies-Bouldin'
    top_db_score['Number of Clusters'] = top_db_score['Number of Clusters'].str.replace('Clusters ', '')
    top_db_score = top_db_score.rename(columns={'DB Score':'Best Score'})

    top_si_score = pd.DataFrame(df.sort_values('Silhouette Score', ascending=False).reset_index().iloc[0,:][['Number of Clusters', 'Silhouette Score']]).T
    top_si_score['Score Type'] = 'Silhouette'
    top_si_score['Number of Clusters'] = top_si_score['Number of Clusters'].str.replace('Clusters ', '')
    top_si_score = top_si_score.rename(columns={'Silhouette Score':'Best Score'})

    top_ch_score = pd.DataFrame(df.sort_values('CH Score', ascending=False).reset_index().iloc[0,:][['Number of Clusters', 'CH Score']]).T
    top_ch_score['Score Type'] = 'Calinski-Harabasz'
    top_ch_score['Number of Clusters'] = top_ch_score['Number of Clusters'].str.replace('Clusters ', '')
    top_ch_score = top_ch_score.rename(columns={'CH Score':'Best Score'})

    top_score = pd.concat([top_db_score, top_si_score, top_ch_score])
    top_score = top_score.rename(columns={'Number of Clusters':'Optimal Number of Clusters'})
    top_score['Clustering Algorithm'] = cluster_technique_name
    top_score['Dataset'] = dataset_name

    return top_score

ahc_scaled_top_scores = top_scores(ahc_scores_scaled, 'Standardized', 'AHC')
ahc_pca_top_scores = top_scores(ahc_scores_pca, 'PCA', 'AHC')
ahc_kernel_top_scores = top_scores(ahc_scores_kernel, 'Kernel PCA', 'AHC')
ahc_top_scores = pd.concat([ahc_scaled_top_scores, ahc_pca_top_scores, ahc_kernel_top_scores])
print(ahc_top_scores.to_latex(index=False))

In [ ]:
def plot_scores(dfs, savepath):
    
    fig, axes = plt.subplots(1, 3, figsize=(30,10))
    axes.flatten()
    
    ax = axes[0]
    ax.plot(range(2,31), dfs[0]['Silhouette Score'], label='Standardized');
    ax.plot(range(2,31), dfs[1]['Silhouette Score'], label='PCA');
    ax.plot(range(2,31), dfs[2]['Silhouette Score'], label='Kernel PCA');
    ax.legend();
    ax.set_ylabel('Score')
    ax.set_xlabel('Number of Clusters')
    ax.set_title('Silhouette Score');

    ax = axes[1]
    ax.plot(range(2,31), dfs[0]['CH Score'], label='Standardized');
    ax.plot(range(2,31), dfs[1]['CH Score'], label='PCA');
    ax.plot(range(2,31), dfs[2]['CH Score'], label='Kernel PCA');
    ax.legend();
    ax.set_ylabel('Score')
    ax.set_xlabel('Number of Clusters')
    ax.set_title('Calinski Harabasz Score');
    
    ax = axes[2]
    ax.plot(range(2,31), dfs[0]['DB Score'], label='Standardized');
    ax.plot(range(2,31), dfs[1]['DB Score'], label='PCA');
    ax.plot(range(2,31), dfs[2]['DB Score'], label='Kernel PCA');
    ax.legend();
    ax.set_ylabel('Score')
    ax.set_xlabel('Number of Clusters')
    ax.set_title('Davies Bouldin Score');
    
    plt.savefig(savepath, dpi=200)
    
plot_scores([ahc_scores_scaled, ahc_scores_pca, ahc_scores_kernel], 'images/scores_ahc.jpg')

### **Fit AHC**

In [ ]:
def fit_ahc(n_clusters, dataframe, dataframe_variables, labels_name):
    
    agglomerativeclustering = AgglomerativeClustering(linkage="ward", n_clusters=n_clusters)
    agglomerativeclustering = agglomerativeclustering.fit(dataframe[dataframe_variables])
    
    df[labels_name] = agglomerativeclustering.labels_
    kernel_principal_df[labels_name] = agglomerativeclustering.labels_
    principal_df[labels_name] = agglomerativeclustering.labels_

In [ ]:
fit_ahc(2, df_scaled, variables, 'scaled_ahc_2')
fit_ahc(3, principal_df, pca_variables, 'pca_ahc_3')
fit_ahc(2, kernel_principal_df, kernel_pca_variables, 'kernel_pca_ahc_2')

### **Maps**

In [ ]:
f, axs = plt.subplots(3, 1, figsize=(40, 40))
axs.flatten()

ax = axs[0]
df.plot(
    column="kernel_pca_ahc_2",
    categorical=True,
    cmap="Set3",
    legend=False,
    linewidth=0,
    ax=ax,
)
ax.set_axis_off()
ax.set_title("AHC for Kernel PCA ($k=2$)", fontsize=30)

ax = axs[1]
df.plot(
    column="pca_ahc_3",
    categorical=True,
    cmap="Set3",
    legend=False,
    linewidth=0,
    ax=ax,
)
ax.set_axis_off()
ax.set_title("AHC for PCA ($k=3$)", fontsize=30)

ax = axs[2]
df.plot(
    column="scaled_ahc_2",
    categorical=True,
    cmap="Set3",
    legend=False,
    linewidth=0,
    ax=ax,
)
ax.set_axis_off()
ax.set_title("AHC for Standardized ($k=2$)", fontsize=30)

plt.tight_layout()

plt.savefig('images/maps_ahc.jpg', dpi=200)

# **REGIONALIZATION**

## **Automatic Zoning Procedure (AZP)**

### **Scores & Optimal Number of Clusters**

In [ ]:
def get_azp_scores(dataframe, variables_list):
    
    azp_clusters = pd.DataFrame()

    for i in range(2, 11):
        
        n_clusters = i
        
        azp = AZP(dataframe, w, variables_list, n_clusters, random_state=42)
        azp.solve()

        azp_clusters['Clusters ' + str(i)] = azp.labels_
        
    
    silhouette_scores = []
    ch_scores = []
    db_scores = []

    for number_clusters in ["Clusters " + str(i) for i in range(2,11)]:
        silhouette_score = metrics.silhouette_score(dataframe[variables_list], azp_clusters[number_clusters])
        silhouette_scores.append((number_clusters, silhouette_score))

        ch_score = metrics.calinski_harabasz_score(dataframe[variables_list], azp_clusters[number_clusters])
        ch_scores.append((number_clusters, ch_score))
        
        db_score = metrics.davies_bouldin_score(dataframe[variables_list], azp_clusters[number_clusters])
        db_scores.append((number_clusters, db_score))


    silhouette_scores = pd.DataFrame(silhouette_scores, columns=['Number of Clusters', 'Silhouette Score']).set_index('Number of Clusters')
    ch_scores = pd.DataFrame(ch_scores, columns=['Number of Clusters', 'CH Score']).set_index('Number of Clusters')
    db_scores = pd.DataFrame(db_scores, columns=['Number of Clusters', 'DB Score']).set_index('Number of Clusters')
    
    scores = silhouette_scores.merge(ch_scores, on=['Number of Clusters']).merge(db_scores, on=['Number of Clusters'])

    return scores

In [ ]:
azp_scores_scaled = get_azp_scores(df_scaled, variables)
azp_scores_pca = get_azp_scores(principal_df, pca_variables)
azp_scores_kernel = get_azp_scores(kernel_principal_df, kernel_pca_variables)

In [ ]:
def top_scores(df, dataset_name, cluster_technique_name):

    top_db_score = pd.DataFrame(df.sort_values('DB Score', ascending=True).reset_index().iloc[0,:][['Number of Clusters', 'DB Score']]).T
    top_db_score['Score Type'] = 'Davies-Bouldin'
    top_db_score['Number of Clusters'] = top_db_score['Number of Clusters'].str.replace('Clusters ', '')
    top_db_score = top_db_score.rename(columns={'DB Score':'Best Score'})

    top_si_score = pd.DataFrame(df.sort_values('Silhouette Score', ascending=False).reset_index().iloc[0,:][['Number of Clusters', 'Silhouette Score']]).T
    top_si_score['Score Type'] = 'Silhouette'
    top_si_score['Number of Clusters'] = top_si_score['Number of Clusters'].str.replace('Clusters ', '')
    top_si_score = top_si_score.rename(columns={'Silhouette Score':'Best Score'})

    top_ch_score = pd.DataFrame(df.sort_values('CH Score', ascending=False).reset_index().iloc[0,:][['Number of Clusters', 'CH Score']]).T
    top_ch_score['Score Type'] = 'Calinski-Harabasz'
    top_ch_score['Number of Clusters'] = top_ch_score['Number of Clusters'].str.replace('Clusters ', '')
    top_ch_score = top_ch_score.rename(columns={'CH Score':'Best Score'})

    top_score = pd.concat([top_db_score, top_si_score, top_ch_score])
    top_score = top_score.rename(columns={'Number of Clusters':'Optimal Number of Clusters'})
    top_score['Clustering Algorithm'] = cluster_technique_name
    top_score['Dataset'] = dataset_name
    
    return top_score

azp_scaled_top_scores = top_scores(azp_scores_scaled, 'Standardized', 'AZP')
azp_pca_top_scores = top_scores(azp_scores_pca, 'PCA', 'AZP')
azp_kernel_top_scores = top_scores(azp_scores_kernel, 'Kernel PCA', 'AZP')
azp_top_scores = pd.concat([azp_scaled_top_scores, azp_pca_top_scores, azp_kernel_top_scores])
print(azp_top_scores.to_latex(index=False))

In [ ]:
def plot_scores(dfs, savepath):
    
    fig, axes = plt.subplots(1, 3, figsize=(30,10))
    axes.flatten()
    
    ax = axes[0]
    ax.plot(range(2,11), dfs[0]['Silhouette Score'], label='Standardized');
    ax.plot(range(2,11), dfs[1]['Silhouette Score'], label='PCA');
    ax.plot(range(2,11), dfs[2]['Silhouette Score'], label='Kernel PCA');
    ax.legend();
    ax.set_ylabel('Score')
    ax.set_xlabel('Number of Clusters')
    ax.set_title('Silhouette Score');

    ax = axes[1]
    ax.plot(range(2,11), dfs[0]['CH Score'], label='Standardized');
    ax.plot(range(2,11), dfs[1]['CH Score'], label='PCA');
    ax.plot(range(2,11), dfs[2]['CH Score'], label='Kernel PCA');
    ax.legend();
    ax.set_ylabel('Score')
    ax.set_xlabel('Number of Clusters')
    ax.set_title('Calinski Harabasz Score');
    
    ax = axes[2]
    ax.plot(range(2,11), dfs[0]['DB Score'], label='Standardized');
    ax.plot(range(2,11), dfs[1]['DB Score'], label='PCA');
    ax.plot(range(2,11), dfs[2]['DB Score'], label='Kernel PCA');
    ax.legend();
    ax.set_ylabel('Score')
    ax.set_xlabel('Number of Clusters')
    ax.set_title('Davies Bouldin Score');
    
    plt.savefig(savepath, dpi=200)
    
plot_scores([azp_scores_scaled, azp_scores_pca, azp_scores_kernel], 'images/scores_azp.jpg')

### **Fit AZP**

In [ ]:
def fit_azp(n_clusters, dataframe, dataframe_variables, labels_name):
    
    azp = AZP(dataframe, w, dataframe_variables, n_clusters, random_state=42)
    azp.solve()
    
    df[labels_name] = azp.labels_
    kernel_principal_df[labels_name] = azp.labels_
    principal_df[labels_name] = azp.labels_

In [ ]:
fit_azp(3, df_scaled, variables, 'scaled_azp_3')
fit_azp(3, principal_df, pca_variables, 'pca_azp_3')
fit_azp(10, kernel_principal_df, kernel_pca_variables, 'kernel_azp_10')

### **Maps**

In [ ]:
f, axs = plt.subplots(3, 1, figsize=(40, 40))
axs.flatten()

ax = axs[0]
df.plot(
    column="kernel_azp_10",
    categorical=True,
    cmap="Set3",
    legend=False,
    linewidth=0,
    ax=ax,
)
ax.set_axis_off()
ax.set_title("AZP for Kernel PCA ($k=10$)", fontsize=30)

ax = axs[1]
df.plot(
    column="pca_azp_3",
    categorical=True,
    cmap="Set3",
    legend=False,
    linewidth=0,
    ax=ax,
)
ax.set_axis_off()
ax.set_title("AZP for PCA ($k=3$)", fontsize=30)

ax = axs[2]
df.plot(
    column="scaled_azp_3",
    categorical=True,
    cmap="Set3",
    legend=False,
    linewidth=0,
    ax=ax,
)
ax.set_axis_off()
ax.set_title("AZP for Standardized ($k=3$)", fontsize=30)

plt.tight_layout()

plt.savefig('images/maps_azp.jpg', dpi=200)

## **Max P**

### **Scores & Optimal Threshold**

In [ ]:
def get_maxp_scores(dataframe, variables_list):
    
    dataframe['count'] = 1
    threshold_name = 'count'
    top_n = 2
    
    maxp_threshold = pd.DataFrame()

    for i in range(2, 31):
        
        threshold = i
        
        maxp = MaxPHeuristic(dataframe, w, variables_list, threshold_name, threshold, top_n, policy='multiple')
        maxp.solve()

        maxp_threshold['Threshold ' + str(i) + ' Clusters'] = maxp.labels_
        
    
    silhouette_scores = []
    ch_scores = []
    db_scores = []

    for threshold in ["Threshold " + str(i) + ' Clusters' for i in range(2,31)]:
        silhouette_score = metrics.silhouette_score(dataframe[variables_list], maxp_threshold[threshold])
        silhouette_scores.append((threshold, silhouette_score))

        ch_score = metrics.calinski_harabasz_score(dataframe[variables_list], maxp_threshold[threshold])
        ch_scores.append((threshold, ch_score))
        
        db_score = metrics.davies_bouldin_score(dataframe[variables_list], maxp_threshold[threshold])
        db_scores.append((threshold, db_score))


    silhouette_scores = pd.DataFrame(silhouette_scores, columns=['Threshold Number', 'Silhouette Score']).set_index('Threshold Number')
    ch_scores = pd.DataFrame(ch_scores, columns=['Threshold Number', 'CH Score']).set_index('Threshold Number')
    db_scores = pd.DataFrame(db_scores, columns=['Threshold Number', 'DB Score']).set_index('Threshold Number')

    scores = silhouette_scores.merge(ch_scores, on=['Threshold Number']).merge(db_scores, on=['Threshold Number'])

    return scores

In [ ]:
maxp_scores_scaled = get_maxp_scores(df_scaled, variables)
maxp_scores_pca = get_maxp_scores(principal_df, pca_variables)
maxp_scores_kernel = get_maxp_scores(kernel_principal_df, kernel_pca_variables)

In [ ]:
def top_scores(df, dataset_name, cluster_technique_name):

    top_db_score = pd.DataFrame(df.sort_values('DB Score', ascending=True).reset_index().iloc[0,:][['Threshold Number', 'DB Score']]).T
    top_db_score['Score Type'] = 'Davies-Bouldin'
    top_db_score['Threshold Number'] = top_db_score['Threshold Number'].str.replace(' Clusters', '')
    top_db_score['Threshold Number'] = top_db_score['Threshold Number'].str.replace('Threshold ', '')    
    top_db_score = top_db_score.rename(columns={'DB Score':'Best Score'})

    top_si_score = pd.DataFrame(df.sort_values('Silhouette Score', ascending=False).reset_index().iloc[0,:][['Threshold Number', 'Silhouette Score']]).T
    top_si_score['Score Type'] = 'Silhouette'
    top_si_score['Threshold Number'] = top_si_score['Threshold Number'].str.replace(' Clusters', '')
    top_si_score['Threshold Number'] = top_si_score['Threshold Number'].str.replace('Threshold ', '')    
    top_si_score = top_si_score.rename(columns={'Silhouette Score':'Best Score'})

    top_ch_score = pd.DataFrame(df.sort_values('CH Score', ascending=False).reset_index().iloc[0,:][['Threshold Number', 'CH Score']]).T
    top_ch_score['Score Type'] = 'Calinski-Harabasz'
    top_ch_score['Threshold Number'] = top_ch_score['Threshold Number'].str.replace(' Clusters', '')
    top_ch_score['Threshold Number'] = top_ch_score['Threshold Number'].str.replace('Threshold ', '')    
    top_ch_score = top_ch_score.rename(columns={'CH Score':'Best Score'})

    top_score = pd.concat([top_db_score, top_si_score, top_ch_score])
    top_score = top_score.rename(columns={'Threshold Number':'Optimal Threshold Level'})
    top_score['Clustering Algorithm'] = cluster_technique_name
    top_score['Dataset'] = dataset_name

    return top_score

maxp_scaled_top_scores = top_scores(maxp_scores_scaled, 'Standardized', 'Max-P')
maxp_pca_top_scores = top_scores(maxp_scores_pca, 'PCA', 'Max-P')
maxp_kernel_top_scores = top_scores(maxp_scores_kernel, 'Kernel PCA', 'Max-P')
maxp_top_scores = pd.concat([maxp_scaled_top_scores, maxp_pca_top_scores, maxp_kernel_top_scores])
print(maxp_top_scores.to_latex(index=False))

In [ ]:
def plot_scores(dfs, savepath):
    
    fig, axes = plt.subplots(1, 3, figsize=(30,10))
    axes.flatten()
    
    ax = axes[0]
    ax.plot(range(2,31), dfs[0]['Silhouette Score'], label='Standardized');
    ax.plot(range(2,31), dfs[1]['Silhouette Score'], label='PCA');
    ax.plot(range(2,31), dfs[2]['Silhouette Score'], label='Kernel PCA');
    ax.legend();
    ax.set_ylabel('Score')
    ax.set_xlabel('Threshold Value')
    ax.set_title('Silhouette Score');

    ax = axes[1]
    ax.plot(range(2,31), dfs[0]['CH Score'], label='Standardized');
    ax.plot(range(2,31), dfs[1]['CH Score'], label='PCA');
    ax.plot(range(2,31), dfs[2]['CH Score'], label='Kernel PCA');
    ax.legend();
    ax.set_ylabel('Score')
    ax.set_xlabel('Threshold Value')
    ax.set_title('Calinski Harabasz Score');
    
    ax = axes[2]
    ax.plot(range(2,31), dfs[0]['DB Score'], label='Standardized');
    ax.plot(range(2,31), dfs[1]['DB Score'], label='PCA');
    ax.plot(range(2,31), dfs[2]['DB Score'], label='Kernel PCA');
    ax.legend();
    ax.set_ylabel('Score')
    ax.set_xlabel('Threshold Value')
    ax.set_title('Davies Bouldin Score');
    
    plt.savefig(savepath, dpi=200)
    
plot_scores([maxp_scores_scaled, maxp_scores_pca, maxp_scores_kernel], 'images/scores_maxp.jpg')

### **Fit MaxP**

In [ ]:
def fit_maxp(threshold, dataframe, dataframe_variables, labels_name):
    
    top_n = 2
    
    dataframe['count'] = 1
    threshold_name = 'count'
    
    maxp = MaxPHeuristic(dataframe, w, dataframe_variables, threshold_name, threshold, top_n, policy='multiple')
    maxp.solve()
    
    df[labels_name] = maxp.labels_
    kernel_principal_df[labels_name] = maxp.labels_
    principal_df[labels_name] = maxp.labels_

In [ ]:
fit_maxp(30,df_scaled, variables, 'scaled_maxp_30')
fit_maxp(30,principal_df, pca_variables, 'pca_maxp_30')
fit_maxp(30,kernel_principal_df, kernel_pca_variables, 'kernel_maxp_30')

### **Maps**

In [ ]:
f, axs = plt.subplots(3, 1, figsize=(40, 40))
axs.flatten()

ax = axs[0]
df.plot(
    column="kernel_maxp_30",
    categorical=True,
    cmap="Set3",
    legend=False,
    linewidth=0,
    ax=ax,
)
ax.set_axis_off()
ax.set_title("Max-P for Kernel PCA ($threshold=30$)", fontsize=30)

ax = axs[1]
df.plot(
    column="pca_maxp_30",
    categorical=True,
    cmap="Set3",
    legend=False,
    linewidth=0,
    ax=ax,
)
ax.set_axis_off()
ax.set_title("Max-P for PCA ($threshold=30$)", fontsize=30)

ax = axs[2]
df.plot(
    column="scaled_maxp_30",
    categorical=True,
    cmap="Set3",
    legend=False,
    linewidth=0,
    ax=ax,
)
ax.set_axis_off()
ax.set_title("Max-P for Standardized ($threshold=30$)", fontsize=30)

plt.tight_layout()

plt.savefig('images/maps_maxp.jpg', dpi=200)

## **SKATER**

### **Scores & Optimal Number of Clusters**

In [ ]:
def get_skater_scores(dataframe, variables_list):
    
    floor = 15
    trace = False
    islands = "increase"
    spanning_forest_kwds = dict(dissimilarity=skm.manhattan_distances, affinity=None, reduction=np.sum, center=np.mean)

    skater_clusters = pd.DataFrame()

    for i in range(2, 31):
        
        n_clusters = i
        
        skater = Skater(dataframe, w, variables_list, n_clusters, floor, trace, islands, spanning_forest_kwds)
        skater.solve()

        skater_clusters['Clusters ' + str(i)] = skater.labels_
        
    
    silhouette_scores = []
    ch_scores = []
    db_scores = []

    for number_clusters in ["Clusters " + str(i) for i in range(2,31)]:
        silhouette_score = metrics.silhouette_score(dataframe[variables_list], skater_clusters[number_clusters])
        silhouette_scores.append((number_clusters, silhouette_score))

        ch_score = metrics.calinski_harabasz_score(dataframe[variables_list], skater_clusters[number_clusters])
        ch_scores.append((number_clusters, ch_score))
        
        db_score = metrics.davies_bouldin_score(dataframe[variables_list], skater_clusters[number_clusters])
        db_scores.append((number_clusters, db_score))


    silhouette_scores = pd.DataFrame(silhouette_scores, columns=['Number of Clusters', 'Silhouette Score']).set_index('Number of Clusters')
    ch_scores = pd.DataFrame(ch_scores, columns=['Number of Clusters', 'CH Score']).set_index('Number of Clusters')
    db_scores = pd.DataFrame(db_scores, columns=['Number of Clusters', 'DB Score']).set_index('Number of Clusters')
    
    scores = silhouette_scores.merge(ch_scores, on=['Number of Clusters']).merge(db_scores, on=['Number of Clusters'])

    return scores

In [ ]:
skater_scores_scaled = get_skater_scores(df_scaled, variables)
skater_scores_pca = get_skater_scores(principal_df, pca_variables)
skater_scores_kernel = get_skater_scores(kernel_principal_df, kernel_pca_variables)

In [ ]:
def top_scores(df, dataset_name, cluster_technique_name):

    top_db_score = pd.DataFrame(df.sort_values('DB Score', ascending=True).reset_index().iloc[0,:][['Number of Clusters', 'DB Score']]).T
    top_db_score['Score Type'] = 'Davies-Bouldin'
    top_db_score['Number of Clusters'] = top_db_score['Number of Clusters'].str.replace('Clusters ', '')
    top_db_score = top_db_score.rename(columns={'DB Score':'Best Score'})

    top_si_score = pd.DataFrame(df.sort_values('Silhouette Score', ascending=False).reset_index().iloc[0,:][['Number of Clusters', 'Silhouette Score']]).T
    top_si_score['Score Type'] = 'Silhouette'
    top_si_score['Number of Clusters'] = top_si_score['Number of Clusters'].str.replace('Clusters ', '')
    top_si_score = top_si_score.rename(columns={'Silhouette Score':'Best Score'})

    top_ch_score = pd.DataFrame(df.sort_values('CH Score', ascending=False).reset_index().iloc[0,:][['Number of Clusters', 'CH Score']]).T
    top_ch_score['Score Type'] = 'Calinski-Harabasz'
    top_ch_score['Number of Clusters'] = top_ch_score['Number of Clusters'].str.replace('Clusters ', '')
    top_ch_score = top_ch_score.rename(columns={'CH Score':'Best Score'})

    top_score = pd.concat([top_db_score, top_si_score, top_ch_score])
    top_score = top_score.rename(columns={'Number of Clusters':'Optimal Number of Clusters'})
    top_score['Clustering Algorithm'] = cluster_technique_name
    top_score['Dataset'] = dataset_name
    
    return top_score

skater_scaled_top_scores = top_scores(skater_scores_scaled, 'Standardized', 'SKATER')
skater_pca_top_scores = top_scores(skater_scores_pca, 'PCA', 'SKATER')
skater_kernel_top_scores = top_scores(skater_scores_kernel, 'Kernel PCA', 'SKATER')
skater_top_scores = pd.concat([skater_scaled_top_scores, skater_pca_top_scores, skater_kernel_top_scores])
print(skater_top_scores.to_latex(index=False))

In [ ]:
def plot_scores(dfs, savepath):
    
    fig, axes = plt.subplots(1, 3, figsize=(30,10))
    axes.flatten()
    
    ax = axes[0]
    ax.plot(range(2,31), dfs[0]['Silhouette Score'], label='Standardized');
    ax.plot(range(2,31), dfs[1]['Silhouette Score'], label='PCA');
    ax.plot(range(2,31), dfs[2]['Silhouette Score'], label='Kernel PCA');
    ax.legend();
    ax.set_ylabel('Score')
    ax.set_xlabel('Number of Clusters')
    ax.set_title('Silhouette Score');

    ax = axes[1]
    ax.plot(range(2,31), dfs[0]['CH Score'], label='Standardized');
    ax.plot(range(2,31), dfs[1]['CH Score'], label='PCA');
    ax.plot(range(2,31), dfs[2]['CH Score'], label='Kernel PCA');
    ax.legend();
    ax.set_ylabel('Score')
    ax.set_xlabel('Number of Clusters')
    ax.set_title('Calinski Harabasz Score');
    
    ax = axes[2]
    ax.plot(range(2,31), dfs[0]['DB Score'], label='Standardized');
    ax.plot(range(2,31), dfs[1]['DB Score'], label='PCA');
    ax.plot(range(2,31), dfs[2]['DB Score'], label='Kernel PCA');
    ax.legend();
    ax.set_ylabel('Score')
    ax.set_xlabel('Number of Clusters')
    ax.set_title('Davies Bouldin Score');
    
    plt.savefig(savepath, dpi=200)
    
plot_scores([skater_scores_scaled, skater_scores_pca, skater_scores_kernel], 'images/scores_skater.jpg')

### **Fit SKATER**

In [ ]:
def fit_skater(n_clusters, dataframe, dataframe_variables, labels_name):
    
    floor = 15
    trace = False
    islands = "increase"
    spanning_forest_kwds = dict(dissimilarity=skm.manhattan_distances, affinity=None, reduction=np.sum, center=np.mean)
    
    skater = Skater(dataframe, w, dataframe_variables, n_clusters, floor, trace, islands, spanning_forest_kwds)
    skater.solve()
    
    df[labels_name] = skater.labels_
    kernel_principal_df[labels_name] = skater.labels_
    principal_df[labels_name] = skater.labels_

In [ ]:
fit_skater(2,df_scaled, variables, 'scaled_skater_2')
fit_skater(2,principal_df, pca_variables, 'pca_skater_2')
fit_skater(3,kernel_principal_df, kernel_pca_variables, 'kernel_skater_3')

In [ ]:
fit_skater(11,df_scaled, variables, 'scaled_skater_11')
fit_skater(13,principal_df, pca_variables, 'pca_skater_13')
fit_skater(9,kernel_principal_df, kernel_pca_variables, 'kernel_skater_9')

### **Maps**

In [ ]:
f, axs = plt.subplots(3, 1, figsize=(40, 40))
axs.flatten()

ax = axs[0]
df.plot(
    column="kernel_skater_3",
    categorical=True,
    cmap="Set3",
    legend=False,
    linewidth=0,
    ax=ax,
)
ax.set_axis_off()
ax.set_title("SKATER for Kernel PCA ($k=3$)", fontsize=30)

ax = axs[1]
df.plot(
    column="pca_skater_2",
    categorical=True,
    cmap="Set3",
    legend=False,
    linewidth=0,
    ax=ax,
)
ax.set_axis_off()
ax.set_title("SKATER for PCA ($k=2$)", fontsize=30)

ax = axs[2]
df.plot(
    column="scaled_skater_2",
    categorical=True,
    cmap="Set3",
    legend=False,
    linewidth=0,
    ax=ax,
)
ax.set_axis_off()
ax.set_title("SKATER for Standardized ($k=2$)", fontsize=30)

plt.tight_layout()

plt.savefig('images/maps_skater.jpg', dpi=200)

In [ ]:
f, axs = plt.subplots(3, 1, figsize=(40, 40))
axs.flatten()

ax = axs[0]
df.plot(
    column="kernel_skater_9",
    categorical=True,
    cmap="Set3",
    legend=False,
    linewidth=0,
    ax=ax,
)
ax.set_axis_off()
ax.set_title("SKATER for Kernel PCA ($k=9$)", fontsize=30)

ax = axs[1]
df.plot(
    column="pca_skater_13",
    categorical=True,
    cmap="Set3",
    legend=False,
    linewidth=0,
    ax=ax,
)
ax.set_axis_off()
ax.set_title("SKATER for PCA ($k=13$)", fontsize=30)

ax = axs[2]
df.plot(
    column="scaled_skater_11",
    categorical=True,
    cmap="Set3",
    legend=False,
    linewidth=0,
    ax=ax,
)
ax.set_axis_off()
ax.set_title("SKATER for Standardized ($k=11$)", fontsize=30)

plt.tight_layout()

plt.savefig('images/maps_skater_2.jpg', dpi=200)

## **Spatial Agglomerative Hierarchical (Spatial AHC)**

### **Scores & Optimal Number of Clusters**

In [ ]:
def get_ward_scores(dataframe, variables_list):
    

    ward_clusters = pd.DataFrame()

    for i in range(2, 31):
                
        ward = WardSpatial(principal_df, w, pca_variables, i)
        ward.solve()

        ward_clusters['Clusters ' + str(i)] = ward.labels_
        
    
    silhouette_scores = []
    ch_scores = []
    db_scores = []

    for number_clusters in ["Clusters " + str(i) for i in range(2,31)]:
        silhouette_score = metrics.silhouette_score(dataframe[variables_list], ward_clusters[number_clusters])
        silhouette_scores.append((number_clusters, silhouette_score))

        ch_score = metrics.calinski_harabasz_score(dataframe[variables_list], ward_clusters[number_clusters])
        ch_scores.append((number_clusters, ch_score))
        
        db_score = metrics.davies_bouldin_score(dataframe[variables_list], ward_clusters[number_clusters])
        db_scores.append((number_clusters, db_score))


    silhouette_scores = pd.DataFrame(silhouette_scores, columns=['Number of Clusters', 'Silhouette Score']).set_index('Number of Clusters')
    ch_scores = pd.DataFrame(ch_scores, columns=['Number of Clusters', 'CH Score']).set_index('Number of Clusters')
    db_scores = pd.DataFrame(db_scores, columns=['Number of Clusters', 'DB Score']).set_index('Number of Clusters')
    
    scores = silhouette_scores.merge(ch_scores, on=['Number of Clusters']).merge(db_scores, on=['Number of Clusters'])

    return scores

In [ ]:
ward_scores_scaled = get_ward_scores(df_scaled, variables)
ward_scores_pca = get_ward_scores(principal_df, pca_variables)
ward_scores_kernel = get_ward_scores(kernel_principal_df, kernel_pca_variables)

In [ ]:
def top_scores(df, dataset_name, cluster_technique_name):

    top_db_score = pd.DataFrame(df.sort_values('DB Score', ascending=True).reset_index().iloc[0,:][['Number of Clusters', 'DB Score']]).T
    top_db_score['Score Type'] = 'Davies-Bouldin'
    top_db_score['Number of Clusters'] = top_db_score['Number of Clusters'].str.replace('Clusters ', '')
    top_db_score = top_db_score.rename(columns={'DB Score':'Best Score'})

    top_si_score = pd.DataFrame(df.sort_values('Silhouette Score', ascending=False).reset_index().iloc[0,:][['Number of Clusters', 'Silhouette Score']]).T
    top_si_score['Score Type'] = 'Silhouette'
    top_si_score['Number of Clusters'] = top_si_score['Number of Clusters'].str.replace('Clusters ', '')
    top_si_score = top_si_score.rename(columns={'Silhouette Score':'Best Score'})

    top_ch_score = pd.DataFrame(df.sort_values('CH Score', ascending=False).reset_index().iloc[0,:][['Number of Clusters', 'CH Score']]).T
    top_ch_score['Score Type'] = 'Calinski-Harabasz'
    top_ch_score['Number of Clusters'] = top_ch_score['Number of Clusters'].str.replace('Clusters ', '')
    top_ch_score = top_ch_score.rename(columns={'CH Score':'Best Score'})

    top_score = pd.concat([top_db_score, top_si_score, top_ch_score])
    top_score = top_score.rename(columns={'Number of Clusters':'Optimal Number of Clusters'})
    top_score['Clustering Algorithm'] = cluster_technique_name
    top_score['Dataset'] = dataset_name
    
    return top_score

ward_scaled_top_scores = top_scores(ward_scores_scaled, 'Standardized', 'Spatial AHC')
ward_pca_top_scores = top_scores(ward_scores_pca, 'PCA', 'Spatial AHC')
ward_kernel_top_scores = top_scores(ward_scores_kernel, 'Kernel PCA', 'Spatial AHC')
ward_top_scores = pd.concat([ward_scaled_top_scores, ward_pca_top_scores, ward_kernel_top_scores])
print(ward_top_scores.to_latex(index=False))

In [ ]:
def plot_scores(dfs, savepath):
    
    fig, axes = plt.subplots(1, 3, figsize=(30,10))
    axes.flatten()
    
    ax = axes[0]
    ax.plot(range(2,31), dfs[0]['Silhouette Score'], label='Standardized');
    ax.plot(range(2,31), dfs[1]['Silhouette Score'], label='PCA');
    ax.plot(range(2,31), dfs[2]['Silhouette Score'], label='Kernel PCA');
    ax.legend();
    ax.set_ylabel('Score')
    ax.set_xlabel('Number of Clusters')
    ax.set_title('Silhouette Score');

    ax = axes[1]
    ax.plot(range(2,31), dfs[0]['CH Score'], label='Standardized');
    ax.plot(range(2,31), dfs[1]['CH Score'], label='PCA');
    ax.plot(range(2,31), dfs[2]['CH Score'], label='Kernel PCA');
    ax.legend();
    ax.set_ylabel('Score')
    ax.set_xlabel('Number of Clusters')
    ax.set_title('Calinski Harabasz Score');
    
    ax = axes[2]
    ax.plot(range(2,31), dfs[0]['DB Score'], label='Standardized');
    ax.plot(range(2,31), dfs[1]['DB Score'], label='PCA');
    ax.plot(range(2,31), dfs[2]['DB Score'], label='Kernel PCA');
    ax.legend();
    ax.set_ylabel('Score')
    ax.set_xlabel('Number of Clusters')
    ax.set_title('Davies Bouldin Score');
    
    plt.savefig(savepath, dpi=200)
    
    
plot_scores([ward_scores_scaled, ward_scores_pca, ward_scores_kernel], 'images/scores_spatial_ahc.jpg')

### **Fit Spatial AHC**

In [ ]:
def fit_ward(n_clusters, dataframe, dataframe_variables, labels_name):
    
    ward = WardSpatial(dataframe, w, dataframe_variables, n_clusters)
    ward.solve()
    
    df[labels_name] = ward.labels_
    kernel_principal_df[labels_name] = ward.labels_
    principal_df[labels_name] = ward.labels_

In [ ]:
fit_ward(2, df_scaled, variables, 'scaled_ward_2')
fit_ward(2, principal_df, pca_variables, 'pca_ward_2')
fit_ward(2, kernel_principal_df, kernel_pca_variables, 'kernel_ward_2')

In [ ]:
fit_ward(30, df_scaled, variables, 'scaled_ward_30')
fit_ward(30, kernel_principal_df, kernel_pca_variables, 'kernel_ward_30')

### **Maps**

In [ ]:
f, axs = plt.subplots(3, 1, figsize=(40, 40))
axs.flatten()

ax = axs[0]
df.plot(
    column="kernel_ward_2",
    categorical=True,
    cmap="Set3",
    legend=False,
    linewidth=0,
    ax=ax,
)
ax.set_axis_off()
ax.set_title("Spatial AHC for Kernel PCA ($k=2$)", fontsize=30)

ax = axs[1]
df.plot(
    column="pca_ward_2",
    categorical=True,
    cmap="Set3",
    legend=False,
    linewidth=0,
    ax=ax,
)
ax.set_axis_off()
ax.set_title("Spatial AHC for PCA ($k=2$)", fontsize=30)

ax = axs[2]
df.plot(
    column="scaled_ward_2",
    categorical=True,
    cmap="Set3",
    legend=False,
    linewidth=0,
    ax=ax,
)
ax.set_axis_off()
ax.set_title("Spatial AHC for Standardized ($k=2$)", fontsize=30)

plt.tight_layout()

plt.savefig('images/maps_spatialahc.jpg', dpi=200)

In [ ]:
f, axs = plt.subplots(2, 1, figsize=(40, 40))
axs.flatten()

ax = axs[0]
df.plot(
    column="kernel_ward_30",
    categorical=True,
    cmap="Set3",
    legend=False,
    linewidth=0,
    ax=ax,
)
ax.set_axis_off()
ax.set_title("Spatial AHC for Kernel PCA ($k=30$)", fontsize=30)


ax = axs[1]
df.plot(
    column="scaled_ward_30",
    categorical=True,
    cmap="Set3",
    legend=False,
    linewidth=0,
    ax=ax,
)
ax.set_axis_off()
ax.set_title("Spatial AHC for Standardized ($k=30$)", fontsize=30)

plt.tight_layout()

plt.savefig('images/maps_spatialahc_2.jpg', dpi=200)

# **DISTRIBUTIONS & SCATTERPLOTS OF CLUSTERS**

## **Distribution of PCs by Cluster**

In [ ]:
def distribution_density_plot(df, variables, cluster_type, savepath):
    
    tidy_df = df.set_index(cluster_type)
    tidy_df = tidy_df[variables[:4]]
    tidy_df = tidy_df.stack()
    tidy_df = tidy_df.reset_index()
    tidy_df = tidy_df.rename(columns={'level_1':'Attribute', 0:'Values'})


    seaborn.set(font_scale=1.5)
    facets = seaborn.FacetGrid(
        data=tidy_df,
        col="Attribute",
        hue=cluster_type,
        sharey=False,
        sharex=False,
        aspect=2,
        col_wrap=4,
    )
    _ = facets.map(seaborn.kdeplot, "Values", shade=True).add_legend()

    plt.savefig(savepath, dpi=200)

In [ ]:
distribution_density_plot(principal_df, pca_variables, 'pca_kmeans_3', 'images/densities_kmeans_pca.jpg')
distribution_density_plot(kernel_principal_df, kernel_pca_variables, 'kernel_pca_kmeans_7', 'images/densities_kmeans_kernel.jpg')

distribution_density_plot(principal_df, pca_variables, 'pca_ahc_3', 'images/densities_ahc_pca.jpg')
distribution_density_plot(kernel_principal_df, kernel_pca_variables, 'kernel_pca_ahc_2', 'images/densities_ahc_kernel.jpg')

distribution_density_plot(principal_df, pca_variables, 'pca_azp_3', 'images/densities_azp_pca.jpg')
distribution_density_plot(kernel_principal_df, kernel_pca_variables, 'kernel_azp_10', 'images/densities_azp_kernel.jpg')

distribution_density_plot(principal_df, pca_variables, 'pca_skater_2', 'images/densities_skater_pca.jpg')
distribution_density_plot(kernel_principal_df, kernel_pca_variables, 'kernel_skater_3', 'images/densities_skater_kernel.jpg')

distribution_density_plot(principal_df, pca_variables, 'pca_ward_2', 'images/densities_spatialahc_pca.jpg')
distribution_density_plot(kernel_principal_df, kernel_pca_variables, 'kernel_ward_2', 'images/densities_spatialahc_kernel.jpg')

distribution_density_plot(principal_df, pca_variables, 'pca_maxp_30', 'images/densities_maxp_pca.jpg')
distribution_density_plot(kernel_principal_df, kernel_pca_variables, 'kernel_maxp_30', 'images/densities_maxp_kernel.jpg')

## **Scatterplots of Data by Cluster**

In [ ]:
fig = plt.figure(figsize=(30,30))

ax = fig.add_subplot(1, 2, 1, projection='3d')
ax.scatter(principal_df['PC1'], principal_df['PC2'], principal_df['PC3'], c=df['pca_kmeans_3'], s=30);
ax.set_title('Principal Components 1-3 with KMeans (k=3)');

ax = fig.add_subplot(1, 2, 2, projection='3d')
ax.scatter(kernel_principal_df['PC1'], kernel_principal_df['PC2'], kernel_principal_df['PC3'], c=df['kernel_pca_kmeans_7'], s=30);
ax.set_title('Kernel Principal Components 1-3 with KMeans (k=7)');

plt.savefig('images/3d_kmeans.jpg', dpi=200)

In [ ]:
fig = plt.figure(figsize=(30,30))

ax = fig.add_subplot(1, 2, 1, projection='3d')
ax.scatter(principal_df['PC1'], principal_df['PC2'], principal_df['PC3'], c=df['pca_ahc_3'], s=30);
ax.set_title('Principal Components 1-3 with AHC (k=3)');

ax = fig.add_subplot(1, 2, 2, projection='3d')
ax.scatter(kernel_principal_df['PC1'], kernel_principal_df['PC2'], kernel_principal_df['PC3'], c=df['kernel_pca_ahc_2'], s=30);
ax.set_title('Kernel Principal Components 1-3 with AHC (k=2)');

plt.savefig('images/3d_ahc.jpg', dpi=200)

In [ ]:
fig = plt.figure(figsize=(30,30))

ax = fig.add_subplot(1, 2, 1, projection='3d')
ax.scatter(principal_df['PC1'], principal_df['PC2'], principal_df['PC3'], c=df['pca_azp_3'], s=30);
ax.set_title('Principal Components 1-3 with AZP (k=3)');

ax = fig.add_subplot(1, 2, 2, projection='3d')
ax.scatter(kernel_principal_df['PC1'], kernel_principal_df['PC2'], kernel_principal_df['PC3'], c=df['kernel_azp_10'], s=30);
ax.set_title('Kernel Principal Components 1-3 with AZP (k=10)');

plt.savefig('images/3d_azp.jpg', dpi=200)

In [ ]:
fig = plt.figure(figsize=(30,30))

ax = fig.add_subplot(1, 2, 1, projection='3d')
ax.scatter(principal_df['PC1'], principal_df['PC2'], principal_df['PC3'], c=df['pca_skater_2'], s=30);
ax.set_title('Principal Components 1-3 with SKATER (k=2)');

ax = fig.add_subplot(1, 2, 2, projection='3d')
ax.scatter(kernel_principal_df['PC1'], kernel_principal_df['PC2'], kernel_principal_df['PC3'], c=df['kernel_skater_3'], s=30);
ax.set_title('Kernel Principal Components 1-3 with SKATER (k=3)');

plt.savefig('images/3d_skater.jpg', dpi=200)

In [ ]:
fig = plt.figure(figsize=(30,30))

ax = fig.add_subplot(1, 2, 1, projection='3d')
ax.scatter(principal_df['PC1'], principal_df['PC2'], principal_df['PC3'], c=df['pca_ward_2'], s=30);
ax.set_title('Principal Components 1-3 with Spatial AHC (k=2)');

ax = fig.add_subplot(1, 2, 2, projection='3d')
ax.scatter(kernel_principal_df['PC1'], kernel_principal_df['PC2'], kernel_principal_df['PC3'], c=df['kernel_ward_2'], s=30);
ax.set_title('Kernel Principal Components 1-3 with Spatial AHC (k=2)');

plt.savefig('images/3d_ward.jpg', dpi=200)

In [ ]:
fig = plt.figure(figsize=(30,30))

ax = fig.add_subplot(1, 2, 1, projection='3d')
ax.scatter(principal_df['PC1'], principal_df['PC2'], principal_df['PC3'], c=df['pca_maxp_30'], s=30);
ax.set_title('Principal Components 1-3 with MaxP (t=30)');

ax = fig.add_subplot(1, 2, 2, projection='3d')
ax.scatter(kernel_principal_df['PC1'], kernel_principal_df['PC2'], kernel_principal_df['PC3'], c=df['kernel_maxp_30'], s=30);
ax.set_title('Kernel Principal Components 1-3 with MaxP (t=30)');

plt.savefig('images/3d_maxp.jpg', dpi=200)

# **METRICS**

## **Isoperimetric Quotient**

In [ ]:
cluster_types = ['scaled_kmeans_3', 'pca_kmeans_3', 'kernel_pca_kmeans_7',
                 'scaled_ahc_2', 'pca_ahc_3', 'kernel_pca_ahc_2',
                 'scaled_azp_3', 'pca_azp_3', 'kernel_azp_10', 
                 'scaled_skater_2', 'pca_skater_2', 'kernel_skater_3',
                 'scaled_skater_11', 'pca_skater_13', 'kernel_skater_9',
                 'scaled_ward_2', 'pca_ward_2', 'kernel_ward_2',
                 'scaled_ward_30', 'kernel_ward_30',
                 'scaled_maxp_30', 'pca_maxp_30', 'kernel_maxp_30']
cluster_type_names = ['K-Means (k=3) with Standardized', 'K-Means (k=3) with PCA', 'K-Means (k=7) with Kernel PCA', 
                      'AHC (k=2) with Standardized', 'AHC (k=3) with PCA', 'AHC (k=2) with Kernel PCA',
                      'AZP (k=3) with Standardized', 'AZP (k=3) with PCA', 'AZP (k=10) with Kernel PCA',
                      'SKATER (k=2) with Standardized', 'SKATER (k=2) with PCA', 'SKATER (k=3) with Kernel PCA',
                      'SKATER (k=11) with Standardized', 'SKATER (k=13) with PCA', 'SKATER (k=9) with Kernel PCA',
                      'Spatial AHC (k=2) with Standardized', 'Spatial AHC (k=2) with PCA', 'Spatial AHC (k=2) with Kernel PCA',
                      'Spatial AHC (k=30) with Standardized', 'Spatial AHC (k=30) with Kernel PCA',
                      'Max-P (t=30) with Standardized', 'Max-P (t=30) with PCA', 'Max-P (t=30) with Kernel PCA']

results = []
for (cluster_type, cluster_type_name) in zip(cluster_types, cluster_type_names): 
    regions = df[[cluster_type, "geometry"]].dissolve(by=cluster_type)
    ipqs = (regions.area * 4 * np.pi / (regions.boundary.length ** 2))
    result = ipqs.to_frame(cluster_type_name)
    results.append(result)

ipq = pd.concat(results, axis=1)
ipq = pd.DataFrame(ipq.mean())
ipq = ipq.rename(columns={0:'IPQ'})
print(ipq.to_latex())

## **Goodness of Fit**

In [ ]:
cluster_types = ['scaled_kmeans_3', 'pca_kmeans_3', 'kernel_pca_kmeans_7',
                 'scaled_ahc_2', 'pca_ahc_3', 'kernel_pca_ahc_2',
                 'scaled_azp_3', 'pca_azp_3', 'kernel_azp_10', 
                 'scaled_skater_2', 'pca_skater_2', 'kernel_skater_3',
                 'scaled_skater_11', 'pca_skater_13', 'kernel_skater_9',
                 'scaled_ward_2', 'pca_ward_2', 'kernel_ward_2',
                 'scaled_ward_30', 'kernel_ward_30',
                 'scaled_maxp_30', 'pca_maxp_30', 'kernel_maxp_30']
cluster_type_names = ['K-Means (k=3) with Standardized', 'K-Means (k=3) with PCA', 'K-Means (k=7) with Kernel PCA', 
                      'AHC (k=2) with Standardized', 'AHC (k=3) with PCA', 'AHC (k=2) with Kernel PCA',
                      'AZP (k=3) with Standardized', 'AZP (k=3) with PCA', 'AZP (k=10) with Kernel PCA',
                      'SKATER (k=2) with Standardized', 'SKATER (k=2) with PCA', 'SKATER (k=3) with Kernel PCA',
                      'SKATER (k=11) with Standardized', 'SKATER (k=13) with PCA', 'SKATER (k=9) with Kernel PCA',
                      'Spatial AHC (k=2) with Standardized', 'Spatial AHC (k=2) with PCA', 'Spatial AHC (k=2) with Kernel PCA',
                      'Spatial AHC (k=30) with Standardized', 'Spatial AHC (k=30) with Kernel PCA',
                      'Max-P (t=30) with Standardized', 'Max-P (t=30) with PCA', 'Max-P (t=30) with Kernel PCA']

ch_scores = []
silhouette_scores = []
db_scores = []

for cluster_type in cluster_types: 
    ch_score = metrics.calinski_harabasz_score(df_scaled[variables], df[cluster_type])
    ch_scores.append(ch_score)
    
    silhouette_score = metrics.silhouette_score(df_scaled[variables], df[cluster_type])
    silhouette_scores.append(silhouette_score)
    
    db_score = metrics.davies_bouldin_score(df_scaled[variables], df[cluster_type])
    db_scores.append(db_score)


scores = pd.DataFrame(list(zip(cluster_type_names, ch_scores, silhouette_scores, db_scores)), columns=['Cluster Type', 'Calinski Harabasz Score', 'Silhouette Score', 'Davies Bouldin Score']).set_index('Cluster Type')
print(scores.to_latex())